In [1]:
%pip install mlflow -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.0/787.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=d2644165-c28d-4426-9d5a-a47f094a264f
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:04<00:00, 81.7MB/s]


In [3]:
import sys

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/')

In [4]:
%pip install mlflow -qq

Note: you may need to restart the kernel to use updated packages.


In [5]:
import logging
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)




In [6]:
LOG_DIR = "./mlruns"


In [7]:
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")
SEED = 42

In [8]:
from sasrec import run_ddp_training, ExperimentConfig

In [9]:
fixed_experiment_parameters = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        n_layers=4,
        d_model=256,
        n_heads=4,
        dropout=0.0,
        log_q_correction=1.0,
        is_cosine_similarity=True,
    ),
    data=ExperimentConfig.DataConfig(
        vocab_size=157_162,
        max_seq_len=100,
        bos=0,
        path_interactions=PATH_INTERACTIONS,
        path_embeddings=PATH_EMBEDDINGS,
        path_artists=PATH_ARTISTS,
        core_min_interaction_per_user=5,
        test_interval_seconds=7 * 24 * 60 * 60,
        max_train_events_per_user=100,
    ),
    tau=None,
    training_dataset=None,
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=None,
    scheduler=ExperimentConfig.SchedulerConfig(
        class_name=None,
        json_args={},
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0,
        eval_every=1,
        logging=True,
        log_dir=LOG_DIR,
        seed=SEED,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100,
    ),
)

In [10]:
from dataclasses import replace
import torch

tau = ExperimentConfig.TauConfig(
    class_name="CosPerUserTau",
    json_args={
        "initial_tau": 0.45,
        "tau_min": None,
        "tau_max": None,
        "num_epochs": 15,
        "num_tokens_per_epoch": 4_019_032,
    },
)

training_dataset = ExperimentConfig.TrainingDatasetConfig(
    batch_size=128,
    device="cuda",
    chunk_rows=64000,
    shuffle=True,
    seed=42,
    pin_memory=True,
    uniform_negative_items=None,
    in_batch_negative_items=None,
)

optimizer = ExperimentConfig.OptimizerConfig(
    class_name="AdamW",
     json_args={
        "lr": 2e-3,
        "weight_decay": 1e-5,
    },
)

for tau_min, tau_max in [(0.04, 0.05), (0.04, 0.055), (0.045, 0.055), (0.045, 0.06)]:
    print(f"Running experiment with tau_min={tau_min} and tau_max={tau_max}...")

    tau.json_args["tau_min"] = tau_min
    tau.json_args["tau_max"] = tau_max

    for uniform, unigram in [(18_000, 12_000), (22_000, 8_000), (26_000, 4_000)]:
        run_ddp_training(
            replace(
                fixed_experiment_parameters, 
                tau=tau,
                training_dataset=replace(
                    training_dataset,
                    uniform_negative_items=uniform, 
                    in_batch_negative_items=unigram,
                ),
                optimizer=optimizer
            ),
            world_size=torch.cuda.device_count()
        )


Running experiment with tau_min=0.04 and tau_max=0.05...


Epochs: 100%|██████████| 15/15 [36:35<00:00, 146.39s/it, train_loss=7.5580]



[Epoch 0] Train Loss: 11.5613 | Validation: hitrate: 0.1273, recall: 0.0348, ndcg: 0.0126, coverage: 0.0020

[Epoch 1] Train Loss: 10.9075 | Validation: hitrate: 0.1506, recall: 0.0409, ndcg: 0.0146, coverage: 0.0039

[Epoch 2] Train Loss: 10.6704 | Validation: hitrate: 0.2155, recall: 0.0650, ndcg: 0.0246, coverage: 0.0102

[Epoch 3] Train Loss: 10.3582 | Validation: hitrate: 0.2567, recall: 0.0799, ndcg: 0.0308, coverage: 0.0247

[Epoch 4] Train Loss: 10.0713 | Validation: hitrate: 0.2786, recall: 0.0862, ndcg: 0.0339, coverage: 0.0560

[Epoch 5] Train Loss: 9.7855 | Validation: hitrate: 0.3016, recall: 0.0963, ndcg: 0.0381, coverage: 0.1332

[Epoch 6] Train Loss: 9.4639 | Validation: hitrate: 0.3213, recall: 0.1047, ndcg: 0.0421, coverage: 0.2046

[Epoch 7] Train Loss: 9.1331 | Validation: hitrate: 0.3252, recall: 0.1074, ndcg: 0.0422, coverage: 0.3461

[Epoch 8] Train Loss: 8.7714 | Validation: hitrate: 0.3355, recall: 0.1114, ndcg: 0.0442, coverage: 0.3959

[Epoch 9] Train Loss: 

Epochs: 100%|██████████| 15/15 [36:55<00:00, 147.73s/it, train_loss=7.1043]



[Epoch 0] Train Loss: 11.1984 | Validation: hitrate: 0.1251, recall: 0.0346, ndcg: 0.0124, coverage: 0.0014

[Epoch 1] Train Loss: 10.5510 | Validation: hitrate: 0.1567, recall: 0.0425, ndcg: 0.0149, coverage: 0.0035

[Epoch 2] Train Loss: 10.3263 | Validation: hitrate: 0.2002, recall: 0.0597, ndcg: 0.0225, coverage: 0.0111

[Epoch 3] Train Loss: 10.0135 | Validation: hitrate: 0.2383, recall: 0.0707, ndcg: 0.0268, coverage: 0.0249

[Epoch 4] Train Loss: 9.7179 | Validation: hitrate: 0.2613, recall: 0.0790, ndcg: 0.0302, coverage: 0.0618

[Epoch 5] Train Loss: 9.4123 | Validation: hitrate: 0.2962, recall: 0.0923, ndcg: 0.0359, coverage: 0.1564

[Epoch 6] Train Loss: 9.0115 | Validation: hitrate: 0.3205, recall: 0.1044, ndcg: 0.0414, coverage: 0.3384

[Epoch 7] Train Loss: 8.5571 | Validation: hitrate: 0.3239, recall: 0.1053, ndcg: 0.0414, coverage: 0.4337

[Epoch 8] Train Loss: 8.1582 | Validation: hitrate: 0.3347, recall: 0.1104, ndcg: 0.0436, coverage: 0.4776

[Epoch 9] Train Loss: 7

...


[Epoch 0] Train Loss: 11.5093 | Validation: hitrate: 0.1257, recall: 0.0347, ndcg: 0.0117, coverage: 0.0021

[Epoch 1] Train Loss: 10.9465 | Validation: hitrate: 0.1619, recall: 0.0440, ndcg: 0.0152, coverage: 0.0034

[Epoch 2] Train Loss: 10.7097 | Validation: hitrate: 0.2114, recall: 0.0619, ndcg: 0.0240, coverage: 0.0110

[Epoch 3] Train Loss: 10.3753 | Validation: hitrate: 0.2507, recall: 0.0766, ndcg: 0.0293, coverage: 0.0252

[Epoch 4] Train Loss: 10.0975 | Validation: hitrate: 0.2743, recall: 0.0837, ndcg: 0.0325, coverage: 0.0563

[Epoch 5] Train Loss: 9.7845 | Validation: hitrate: 0.3043, recall: 0.0966, ndcg: 0.0377, coverage: 0.1437

[Epoch 6] Train Loss: 9.4216 | Validation: hitrate: 0.3254, recall: 0.1077, ndcg: 0.0427, coverage: 0.2790

[Epoch 7] Train Loss: 9.0366 | Validation: hitrate: 0.3400, recall: 0.1147, ndcg: 0.0461, coverage: 0.3270

[Epoch 8] Train Loss: 8.7007 | Validation: hitrate: 0.3411, recall: 0.1148, ndcg: 0.0461, coverage: 0.3885

[Epoch 9] Train Loss: 

Epochs: 100%|██████████| 15/15 [36:46<00:00, 147.12s/it, train_loss=7.4554]



[Epoch 0] Train Loss: 11.1181 | Validation: hitrate: 0.1300, recall: 0.0349, ndcg: 0.0122, coverage: 0.0019

[Epoch 1] Train Loss: 10.5797 | Validation: hitrate: 0.1525, recall: 0.0413, ndcg: 0.0144, coverage: 0.0037

[Epoch 2] Train Loss: 10.3648 | Validation: hitrate: 0.2053, recall: 0.0598, ndcg: 0.0231, coverage: 0.0105

[Epoch 3] Train Loss: 10.0516 | Validation: hitrate: 0.2425, recall: 0.0731, ndcg: 0.0278, coverage: 0.0259

[Epoch 4] Train Loss: 9.7794 | Validation: hitrate: 0.2616, recall: 0.0793, ndcg: 0.0304, coverage: 0.0559

[Epoch 5] Train Loss: 9.4749 | Validation: hitrate: 0.3047, recall: 0.0972, ndcg: 0.0376, coverage: 0.1228

[Epoch 6] Train Loss: 9.1431 | Validation: hitrate: 0.3221, recall: 0.1049, ndcg: 0.0418, coverage: 0.2068

[Epoch 7] Train Loss: 8.8234 | Validation: hitrate: 0.3362, recall: 0.1120, ndcg: 0.0446, coverage: 0.2881

[Epoch 8] Train Loss: 8.5042 | Validation: hitrate: 0.3435, recall: 0.1145, ndcg: 0.0461, coverage: 0.3521

[Epoch 9] Train Loss: 8

Epochs: 100%|██████████| 15/15 [36:48<00:00, 147.24s/it, train_loss=6.8887]



[Epoch 0] Train Loss: 10.5718 | Validation: hitrate: 0.1261, recall: 0.0340, ndcg: 0.0117, coverage: 0.0015

[Epoch 1] Train Loss: 10.0396 | Validation: hitrate: 0.1560, recall: 0.0420, ndcg: 0.0149, coverage: 0.0052

[Epoch 2] Train Loss: 9.8144 | Validation: hitrate: 0.2160, recall: 0.0636, ndcg: 0.0242, coverage: 0.0114

[Epoch 3] Train Loss: 9.4962 | Validation: hitrate: 0.2526, recall: 0.0772, ndcg: 0.0295, coverage: 0.0261

[Epoch 4] Train Loss: 9.2241 | Validation: hitrate: 0.2707, recall: 0.0821, ndcg: 0.0312, coverage: 0.0591

[Epoch 5] Train Loss: 8.9276 | Validation: hitrate: 0.3048, recall: 0.0973, ndcg: 0.0375, coverage: 0.1226

[Epoch 6] Train Loss: 8.6008 | Validation: hitrate: 0.3235, recall: 0.1060, ndcg: 0.0417, coverage: 0.2695

[Epoch 7] Train Loss: 8.2474 | Validation: hitrate: 0.3334, recall: 0.1101, ndcg: 0.0431, coverage: 0.3502

[Epoch 8] Train Loss: 7.8980 | Validation: hitrate: 0.3416, recall: 0.1149, ndcg: 0.0465, coverage: 0.3335

[Epoch 9] Train Loss: 7.6